# 03 — Validación de campo: trazabilidad y estadística
**Paper Cañete · Royal Society Open Science**

Reconstruye la contabilidad de las dos salidas de campo con trazabilidad total
(alerta → GPS → foto → resultado), calcula los IC de Wilson, el test de Fisher,
y el resultado nuevo: **la probabilidad del modelo discriminó en el campo (AUC=0.80)**.

⚠️ **Coordenadas sensibles**: los CSV de campo contienen ubicaciones exactas de evidencias.
No publicar crudos; para el repositorio público generalizar a la celda de 50 m.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact, mannwhitneyu

m1 = pd.read_csv('../datos/alertas_modelo1_85.csv')
m2 = pd.read_csv('../datos/alertas_modelo2_146.csv')
c1 = pd.read_csv('../datos/campo_salida1.csv')
c2 = pd.read_csv('../datos/campo_salida2.csv')
print(f"alertas M1: {len(m1)} | alertas M2: {len(m2)} | campo: {len(c1)}+{len(c2)} puntos")

## 1. Trazabilidad: cada punto de campo → su alerta

In [ ]:
def dist_m(lat1, lon1, lat2, lon2):
    return np.sqrt(((lat2-lat1)*111320)**2 + ((lon2-lon1)*111320*np.cos(np.radians(-13.1)))**2)

def clasificar(pred, es_sitio, d):
    conf = str(es_sitio).strip().lower() not in ('no','nan')
    if str(pred).strip()=='si':
        return 'alerta_confirmada' if conf else 'alerta_descartada_FP'
    return 'hallazgo_adyacente' if d < 110 else 'sitio_no_alertado_independiente'

rows = []
for campo, alertas, salida, pcol in [(c1, m1, 1, 'prioridad'), (c2, m2, 2, 'probabilidad')]:
    for _, r in campo.iterrows():
        d = dist_m(r['latitud'], r['longitud'], alertas.lat, alertas.lon)
        i = int(d.idxmin())
        rows.append(dict(salida=salida, foto=r['foto'], predicho=r['predicho'],
            alerta=int(alertas.loc[i,'alerta_num']), info_alerta=alertas.loc[i,pcol],
            dist_alerta_m=round(float(d.min()),1), resultado=r['es_sitio'],
            categoria=clasificar(r['predicho'], r['es_sitio'], d.min())))
traz = pd.DataFrame(rows)
print(traz.to_string(index=False))
print('\n', traz.categoria.value_counts().to_string())
traz.to_csv('tabla_trazabilidad_31pts.csv', index=False)

## 2. Tasas de confirmación con incertidumbre

In [ ]:
import math
def wilson(k, n, z=1.96):
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return c-h, c+h

conf = lambda s: traz[(traz.salida==s)&(traz.predicho=='si')]
for s, nombre in [(1,'Modelo 1'), (2,'Modelo 2')]:
    t = conf(s); k = (t.categoria=='alerta_confirmada').sum(); n = len(t)
    lo, hi = wilson(k, n)
    print(f"{nombre}: {k}/{n} = {k/n:.1%}  IC95% Wilson [{lo:.1%}, {hi:.1%}]")

k1,n1 = 4,9; k2,n2 = 12,17
odds, p = fisher_exact([[k1,n1-k1],[k2,n2-k2]])
print(f"Fisher exacto M1 vs M2: p = {p:.3f}  -> diferencia NO concluyente; reportar como exploratoria")

## 3. Resultado nuevo: la probabilidad discriminó en el campo (Modelo 2)

In [ ]:
t2 = conf(2).copy()
t2['confirmado'] = (t2.categoria=='alerta_confirmada').astype(int)
p_conf = t2.loc[t2.confirmado==1,'info_alerta'].astype(float)
p_fp   = t2.loc[t2.confirmado==0,'info_alerta'].astype(float)
u, pv = mannwhitneyu(p_conf, p_fp, alternative='greater')
auc = u/(len(p_conf)*len(p_fp))
print(f"prob media confirmadas (n={len(p_conf)}): {p_conf.mean():.3f}")
print(f"prob media descartadas (n={len(p_fp)}):  {p_fp.mean():.3f}")
print(f"Mann-Whitney U={u:.0f}, p={pv:.3f}  ->  AUC de campo = {auc:.2f}")

fig, ax = plt.subplots(figsize=(6.5,4))
rng = np.random.default_rng(0)
for vals, x0, col, lab in [(p_fp, 0, 'steelblue', 'descartada (FP)'), (p_conf, 1, 'firebrick', 'confirmada')]:
    ax.scatter(x0 + rng.uniform(-.08,.08,len(vals)), vals, s=70, alpha=.8, color=col, label=lab, zorder=3)
    ax.hlines(vals.mean(), x0-.18, x0+.18, color=col, lw=2)
ax.set_xticks([0,1]); ax.set_xticklabels(['Alerta descartada','Alerta confirmada'])
ax.set_ylabel('Probabilidad del modelo'); ax.set_title(f'El ranking llevó señal al campo (AUC={auc:.2f}, p={pv:.3f}, n=17)')
ax.legend(); plt.tight_layout(); plt.savefig('fig_auc_campo.png', dpi=200); plt.show()

## 4. Los cinco sitios no alertados

In [ ]:
na = traz[traz.predicho!='si'][['salida','foto','dist_alerta_m','categoria']]
print(na.to_string(index=False))
print('\nDefinición métrica: hallazgo_adyacente = a menos de 110 m (~2 celdas) de una alerta.')

## Para el paper
- Tabla de trazabilidad → material suplementario S1 (con coordenadas generalizadas).
- 44.4% [18.9–73.3] vs 70.6% [46.9–86.7], Fisher p=0.234 → ventaja hidrológica **exploratoria**.
- AUC de campo 0.80 (p=0.032, n=17) → el resultado positivo central: el ranking funcionó
  donde importa, aunque la métrica interna estuviera rota.